# DeepSORT track_id Annotator
This notebook is used to assign track ids using DeepSORT, to detected objects that were previously generated by a pretrained YOLO model and stored in a .json file with a COCO dataset format.

## Imports

In [ ]:
import sys
from pathlib import Path
import json
import cv2

In [ ]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [ ]:
from src.utils.config.config import Config
from src.utils.bbox_utils import BboxUtils
from src.tracker.deepsort_tracker import DeepSortTracker

## Config

In [ ]:
config_loader = Config()
cfg = config_loader.load_config()

raw_dataset = cfg.project_root / config_loader.get("paths", "dataset.raw") / "CafeV1" / "yolo"

bboxUtils = BboxUtils

## Functions

### load_existing_json

In [ ]:
def load_existing_json(json_path):
    if json_path.exists():
        with open(json_path, "r") as f:
            return json.load(f)
    # Empty data
    return None

### save_json

In [ ]:
def save_json(data, json_path):
    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)

### process_clip

In [ ]:
def process_clip(clip_path):
    clip_name = clip_path.name # 0
    
    images_path = clip_path / "images" 
    
    output_path = raw_dataset / "clips" / clip_name
    output_images_path = output_path / "images"  
    output_json_path = output_path / "anns.json"
    
    if not images_path.exists():
        print(f"Couldn't process clip {clip_name}")
        return

    # Get and sort frames
    image_files = (p for p in images_path.iterdir()) # list of frames
    image_files = sorted(image_files, key=lambda x: int(x.stem.split("_")[1]))

    # Ensure path exists
    output_images_path.mkdir(parents=True, exist_ok=True)

    json_data = load_existing_json(output_json_path)
    
    # Check if frame has anns
    if json_data == None:
        print(f"No annotations found for {clip_name}")
        return
    
    # Initalize DeepSORT for each clip
    tracker = DeepSortTracker(cfg.deepsort.max_age,
                          cfg.deepsort.n_init,
                          cfg.deepsort.nn_budget,
                          cfg.deepsort.max_cosine_distance,
                          cfg.deepsort.max_iou_distance,
                          cfg.deepsort.nms_max_overlap
                        )
    
    for image_id, image_file in enumerate(image_files):
        image_path = images_path / image_file
        
        # Read the frame
        frame = cv2.imread(image_path)
        if frame is None:
            continue
    
        frame_anns = [
            ann for ann in json_data["annotations"]
            if ann["image_id"] == image_id
        ]
        
        ann_refs = []
        detections = []
        for ann in frame_anns:
            # skip annotations that already have a track id
            # if ann["attributes"].get("track_id"):
            #     continue 
            
            ann_refs.append(ann)
            
            x1, y1, w, h = ann["bbox"]
            cls_id = int(ann["category_id"])
            
            # tuples of ( [left,top,w,h], confidence, detection_class )
            detections.append(([x1, y1, w, h], 1.0, cls_id))  # fake conf
    
        if len(detections) == 0:
            continue
        
        # Run deepsort
        results = tracker.update(detections, frame)

        for track in results:
            if not track.is_confirmed():
                continue
    
            track_id = int(track.track_id)
            bbox = track.to_ltrb() # x1, y1, x2, y2
            
            best_ann = None
            best_iou = 0
            # find the original bbox
            for ann in ann_refs:
                x, y, w, h = ann["bbox"]
                ax1, ay1 = x, y
                ax2, ay2 = x + w, y + h
                
                score = bboxUtils.get_iou(bbox, [ax1, ay1, ax2, ay2])
                
                if score > best_iou:
                    best_iou = score
                    best_ann = ann

            # assign track id if match is good
            if best_ann is not None and best_iou > 0.5:
                if "attributes" not in best_ann:
                    best_ann["attributes"] = {}
                best_ann["attributes"]["track_id"] = track_id

        # break # temp - used to validate one frame

    save_json(json_data, output_json_path)

## Run

In [ ]:
def main():
    clips_path = raw_dataset / "Clips"
    clip_dirs = sorted(p for p in clips_path.iterdir() if p.is_dir())

    for clip_path in clip_dirs:
        process_clip(clip_path)
        # return # temp - used to validate one clip

main()